# 9.16 模型給機率後，程式怎樣決策？


假設客服只能做兩件事：直接顯示一個教學頁，或先問一個澄清問題。模型覺得應顯示卡片頁的機率是0.8。要不要直接顯示，不能只從0.8決定；顯示錯誤與多問一次的代價也要先約定。

用人工成本說明：答錯損失5單位，答對為0；澄清固定損失1單位，暫假設澄清能解決這一輪不確定性。若 p 是可用來估算正確性的機率，直接回答的期望損失是 `5*(1-p)`，澄清是1。在 p 大於0.8時，前者較小；相等時由事先規則處理。這個門檻只屬於這個簡化成本表，不能拿去自動批准轉帳。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
for p in [0.6, 0.8, 0.9]:
    answer_cost = 5 * (1 - p)
    action = "回答" if answer_cost < 1 - 1e-9 else "澄清"
    print(p, round(answer_cost, 3), action)

程式在0.6、0.8選澄清，在0.9選回答。實際服務若澄清也可能失敗，就要把後續成功機會及成本納入；若模型沒有校準，把集中度當 p 會使期望成本失真。部署前應在 policy 資料上實測規則，最後同時報自動處理覆蓋率、已處理題的錯誤率與總成本，不能只報「留下來的題很準」。

這種安排把工作分成兩層：模型從材料預測，程式依任務成本與授權決定動作。可以在適當時查資料、呼叫工具或轉人工，但每種動作有自己的條件；本節只做有限模擬，不執行真實個人資料操作。第19章的 Auto 也使用這個分工：先判斷資訊是否足夠，再選解題預算。

使用者提供的 DeskMind 是可研究的實務案例。其公開訓練說明列出從成熟模型適配、教師分布與候選重排等做法，適合對照本章的資料契約；它並未提供足以重現 Jev 專有訓練的全部細節。小型自測全對也不能替代按家族留出的校準與泛化評估。我們會使用自製有限世界及隨機初始化權重，讓標籤與限制可以查清楚。

練習：若澄清成本降成0.5，在上述假設下門檻變多少？解 `5*(1-p)<0.5` 得 p>0.9。多問一次越便宜，程式就有理由更常澄清；這是成本選擇，不代表模型忽然變得不確定。

案例：[DeskMind launch](https://deskmind.dev/zh/blog/launch/)、[公開 training 說明](https://github.com/deskmind-ai/brain/blob/main/docs/training.md)。
